# Data Loss Prevention

****(DLP API must be enabled)***

## [DLP Jobs](https://docs.cloud.google.com/sensitive-data-protection/docs/concepts-job-triggers)

A job is an action that Sensitive Data Protection runs to either scan content for sensitive data or calculate the risk of re-identification.

Sensitive Data Protection creates and runs a job resource whenever you tell it to inspect the data.

There are two types of Sensitive Data Protection Jobs:

- *Inspection Jobs*: Inspect your content for sensitive data according to your criteria and generate summary reports of where and what type of sensitive data exists.

- *Risk Analysis Jobs*: Analyze de-identified data and return metrics about the likelihood that the data can be re-identified.

Those jobs can be scheduled using job triggers, and it can scan GCS buckets, BQ tables, and Datastore kinds.

A job trigger is represented in the DLP API by the [**JobTrigger**](https://docs.cloud.google.com/sensitive-data-protection/docs/concepts-job-triggers#the_jobtrigger_object) object.

In [10]:
from google.cloud import dlp_v2
from typing import Optional
import sys

sys.path.append("..")

from agent.core_agent.config.agent_settings import GCPConfig
from dotenv import load_dotenv

load_dotenv()

True

### [InfoTypes](https://docs.cloud.google.com/sensitive-data-protection/docs/infotypes-reference?hl=en)

An InfoType is a type of sensitive data, such as name, email address, telephone humber, credit card number, etc.

Sensitive Data Protection uses [InfoType Detectors](https://docs.cloud.google.com/sensitive-data-protection/docs/concepts-infotypes?hl=en#dlp_inspect_phone_number-python) in the configuration for its scans to determine what to inspect for and how to transform findings. For example, if its necessary to obtain an email address, it is required to specify the EMAIL_ADDRESS infoType detector in the inspection configuration.


Each one of them has a *detector*. Sensitive Data Protection uses InfoType detectors in the configuration for its scans to determine what ti inspect for and how to transform findings

Find the full list of info types [here](https://docs.cloud.google.com/sensitive-data-protection/docs/infotypes-reference?hl=en#general-infotypes).

It is recommended to use general info types instead of specific ones.

### Detection Techniques

To detect content that corresponds to built-in InfoTypes, Sentitive Data Protection uses various techniques including pattern matching, checksum validation, machine learning, and context analysis. For example, to detect CREDIT_CARD_NUMBER infoType, Sensitive Data Protection checks for known issuer prefixes, validates checksums, analyzes character lengths, and considers the context in which the potential credit card number appears.

In [2]:
gcp_config = GCPConfig()

In [13]:
def create_dlp_job(
    project: str,
    bucket: str,
    info_types: list[str],
    job_id: str = None,
    max_findings: int = 100,
    auto_populate_timespan: bool = True,
) -> None:
    """Uses the Data Loss Prevention API to create a DLP job.
    Args:
        project: The project id to use as a parent resource.
        bucket: The name of the GCS bucket to scan. This sample scans all
            files in the bucket.
        info_types: A list of strings representing info types to look for.
            A full list of info type categories can be fetched from the API.
        job_id: The id of the job. If omitted, an id will be randomly generated.
        max_findings: The maximum number of findings to report; 0 = no maximum.
        auto_populate_timespan: Automatically populates time span config start
            and end times in order to scan new content only.
    """

    # Instantiate a client.
    dlp = dlp_v2.DlpServiceClient()

    # Convert the project id into a full resource id.
    parent = f"projects/{project}"

    # Prepare info_types by converting the list of strings into a list of
    # dictionaries (protos are also accepted).
    info_types = [{"name": info_type} for info_type in info_types]

    # Construct the configuration dictionary. Keys which are None may
    # optionally be omitted entirely.
    inspect_config = {
        "info_types": info_types,
        "min_likelihood": dlp_v2.Likelihood.UNLIKELY,
        "limits": {"max_findings_per_request": max_findings},
        "include_quote": True,
    }

    # Construct a cloud_storage_options dictionary with the bucket's URL.
    url = f"gs://{bucket}/*"
    storage_config = {
        "cloud_storage_options": {"file_set": {"url": url}},
        # Time-based configuration for each storage object.
        "timespan_config": {
            # Auto-populate start and end times in order to scan new objects
            # only.
            "enable_auto_population_of_timespan_config": auto_populate_timespan
        },
    }

    # Construct the job definition.
    job = {"inspect_config": inspect_config, "storage_config": storage_config}

    # Call the API.
    response = dlp.create_dlp_job(
        request={"parent": parent, "inspect_job": job, "job_id": job_id}
    )

    # Print out the result.
    print(f"Job : {response.name} status: {response.state}")


In [16]:
info_types = ["FINANCIAL_ID", "CREDIT_CARD_DATA", "GOVERNMENT_ID", "PASSPORT", "PHONE_NUMBER", "SECURITY_DATA",]

create_dlp_job(
    project=gcp_config.PROJECT_ID,
    bucket="ai-hub-sharepoint",
    info_types=info_types,
    job_id="eamadorm_first_job",
)

Job : projects/p-dev-gce-60pf/dlpJobs/i-eamadorm_first_job status: 1


## Get Job

In [21]:
def get_dlp_job(project: str, job_name: str) -> None:
    """Uses the Data Loss Prevention API to retrieve a DLP job.
    Args:
        project: The project id to use as a parent resource.
        job_name: The name of the DlpJob resource to be retrieved.
    """

    # Instantiate a client.
    dlp = dlp_v2.DlpServiceClient()

    # Convert the project id and job name into a full resource id.
    job_name = f"projects/{project}/locations/global/dlpJobs/{job_name}"

    # Call the API
    response = dlp.get_dlp_job(request={"name": job_name})

    print(f"Job: {response.name} Status: {response.state}")

    return response

In [22]:
job_status = get_dlp_job("p-dev-gce-60pf", "i-eamadorm_first_job")

Job: projects/p-dev-gce-60pf/locations/global/dlpJobs/i-eamadorm_first_job Status: 3


In [23]:
job_status

name: "projects/p-dev-gce-60pf/locations/global/dlpJobs/i-eamadorm_first_job"
type_: INSPECT_JOB
state: DONE
inspect_details {
  requested_options {
    snapshot_inspect_template {
    }
    job_config {
      storage_config {
        cloud_storage_options {
          file_set {
            url: "gs://ai-hub-sharepoint/*"
          }
        }
        timespan_config {
          enable_auto_population_of_timespan_config: true
        }
      }
      inspect_config {
        info_types {
          name: "FINANCIAL_ID"
        }
        info_types {
          name: "CREDIT_CARD_DATA"
        }
        info_types {
          name: "GOVERNMENT_ID"
        }
        info_types {
          name: "PASSPORT"
        }
        info_types {
          name: "PHONE_NUMBER"
        }
        info_types {
          name: "SECURITY_DATA"
        }
        min_likelihood: UNLIKELY
        limits {
          max_findings_per_request: 100
        }
        include_quote: true
      }
    }
  }
  result 

## Create Job Trigger

In [ ]:
def create_trigger(
    project: str,
    bucket: str,
    scan_period_days: int,
    info_types: List[str],
    trigger_id: Optional[str] = None,
    display_name: Optional[str] = None,
    description: Optional[str] = None,
    min_likelihood: Optional[int] = None,
    max_findings: Optional[int] = None,
    auto_populate_timespan: Optional[bool] = False,
) -> None:
    """Creates a scheduled Data Loss Prevention API inspect_content trigger.
    Args:
        project: The Google Cloud project id to use as a parent resource.
        bucket: The name of the GCS bucket to scan. This sample scans all
            files in the bucket using a wildcard.
        scan_period_days: How often to repeat the scan, in days.
            The minimum is 1 day.
        info_types: A list of strings representing info types to look for.
            A full list of info type categories can be fetched from the API.
        trigger_id: The id of the trigger. If omitted, an id will be randomly
            generated.
        display_name: The optional display name of the trigger.
        description: The optional description of the trigger.
        min_likelihood: A string representing the minimum likelihood threshold
            that constitutes a match. One of: 'LIKELIHOOD_UNSPECIFIED',
            'VERY_UNLIKELY', 'UNLIKELY', 'POSSIBLE', 'LIKELY', 'VERY_LIKELY'.
        max_findings: The maximum number of findings to report; 0 = no maximum.
        auto_populate_timespan: Automatically populates time span config start
            and end times in order to scan new content only.
    Returns:
        None; the response from the API is printed to the terminal.
    """

    # Instantiate a client.
    dlp = google.cloud.dlp_v2.DlpServiceClient()

    # Prepare info_types by converting the list of strings into a list of
    # dictionaries (protos are also accepted).
    info_types = [{"name": info_type} for info_type in info_types]

    # Construct the configuration dictionary. Keys which are None may
    # optionally be omitted entirely.
    inspect_config = {
        "info_types": info_types,
        "min_likelihood": min_likelihood,
        "limits": {"max_findings_per_request": max_findings},
    }

    # Construct a cloud_storage_options dictionary with the bucket's URL.
    url = f"gs://{bucket}/*"
    storage_config = {
        "cloud_storage_options": {"file_set": {"url": url}},
        # Time-based configuration for each storage object.
        "timespan_config": {
            # Auto-populate start and end times in order to scan new objects
            # only.
            "enable_auto_population_of_timespan_config": auto_populate_timespan
        },
    }

    # Construct the job definition.
    job = {"inspect_config": inspect_config, "storage_config": storage_config}

    # Construct the schedule definition:
    schedule = {
        "recurrence_period_duration": {"seconds": scan_period_days * 60 * 60 * 24}
    }

    # Construct the trigger definition.
    job_trigger = {
        "inspect_job": job,
        "display_name": display_name,
        "description": description,
        "triggers": [{"schedule": schedule}],
        "status": google.cloud.dlp_v2.JobTrigger.Status.HEALTHY,
    }

    # Convert the project id into a full resource id.
    parent = f"projects/{project}"

    # Call the API.
    response = dlp.create_job_trigger(
        request={"parent": parent, "job_trigger": job_trigger, "trigger_id": trigger_id}
    )

    print(f"Successfully created trigger {response.name}")

In [1]:
import sys

sys.path.append("..")

from pipelines.document_classification.pipeline import ClassificationPipeline

In [2]:
pipe = ClassificationPipeline()

In [4]:
pipe.dlp_trigger("gs://enterprise_knowledgebase_landing_zone/TT1 Diseño y simulación de la operación de un sistema híbrido de secado de podas de pino y encino para la generación eléctrica.pdf")

2026-04-16 20:27:02.790 | INFO     | pipelines.document_classification.pipeline:dlp_trigger:42 - Triggering DLP scan for: gs://enterprise_knowledgebase_landing_zone/TT1 Diseño y simulación de la operación de un sistema híbrido de secado de podas de pino y encino para la generación eléctrica.pdf
2026-04-16 20:27:02.791 | INFO     | pipelines.document_classification.dlp_service:inspect_gcs_file:35 - Starting DLP scan for: gs://enterprise_knowledgebase_landing_zone/TT1 Diseño y simulación de la operación de un sistema híbrido de secado de podas de pino y encino para la generación eléctrica.pdf
2026-04-16 20:27:03.523 | INFO     | pipelines.document_classification.dlp_service:inspect_gcs_file:80 - DLP Job created: projects/p-dev-gce-60pf/locations/global/dlpJobs/i-8945611647032256147
2026-04-16 20:27:03.830 | INFO     | pipelines.document_classification.dlp_service:wait_for_job:117 - Waiting for DLP Job... (Current state: PENDING)
2026-04-16 20:27:08.951 | INFO     | pipelines.document_cla

InvalidArgument: 400 Image size 4413988 exceeds limit of 4194304 [reason: "3"
domain: "dlp.googleapis.com"
]